# Week 3 Day 5 Lab 1 (free route) - The Copilot on Render

## Running the Copilot beside the app and the agent

On the AWS route, Lab 1 deploys all of UnderCover again, with the Copilot on AgentCore Runtime beside the claim agent and an AgentCore Memory of its own. On the free route we run the Copilot as a third web service on Render, beside your app and the agent from Day 4, with the context layer inside the same service.

AgentCore Runtime, the Gateway and the Memory are AWS services, so for those you can watch me. The rest of today's build works the same way: the Copilot acts with the sign-in of the handler who asked, shows each tool call in the sidebar as it happens, asks before it pays and keeps each conversation in Supabase.

This notebook has no code to run. It gives you the steps, with links to Render's own documentation.

## 5 steps:

STEP 1: Update the app and the agent on Render

STEP 2: Create the Copilot's web service

STEP 3: Point the app at the Copilot

STEP 4: The handover conversation

STEP 5: Compare with the AWS route

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of today's first lab. The main lab, <a href="week3day5_lab1.ipynb">week3day5_lab1.ipynb</a>, runs on AWS. You need the two Render services from Day 4, and today's build of the product, yours or <code>week3day5-end</code>, pushed to GitHub as on Day 2. Step 2 needs one name from your build, and shows you where to find it.<br/><br/>I do this week on AWS, so I'm not recording the Render steps. Render's website changes from time to time, so if a screen looks different from what's described here, follow the linked page in Render's documentation.<br/><br/>Step 4 works two claims with the agent and asks the Copilot five questions, which can use most of your 50 free requests for the day. Start it on a day you haven't used any. If you run out, the requests come back the next day.
            </span>
        </td>
    </tr>
</table>

## Step 1: Update the app and the agent on Render

Today's build adds the Copilot's routes to the app, and the Copilot's code to the `agent` package beside the claim agent's. So both of your services on Render need today's code.

- **To deploy my build,** open each web service in the [Render Dashboard](https://dashboard.render.com), go to **Settings**, change the **Branch** to `week3day5-end`, and save. Then choose **Manual Deploy** and **Deploy latest commit**.
- **To deploy your own build,** push your branch to your own GitHub repository, as on Day 2. If it's a new branch, change each service's **Branch** to it under **Settings**. Then choose **Manual Deploy** and **Deploy latest commit** on each service.

The build and start commands and the environment variables stay as they were on Day 4. Step 3 adds one more to the app.

Render's documentation: [Deploys](https://render.com/docs/deploys).

## Step 2: Create the Copilot's web service

### Find the Copilot's app

Render starts the Copilot with uvicorn, as it starts the agent, so it needs to know where the Copilot's AgentCore app is, written as `module:name`. The claim agent's is `agent.entrypoint:app`.

Your build chose where the Copilot's app lives, and `package.json` shows it. The script that `npm run dev` uses to serve the Copilot on port 8081 runs `uvicorn` with the Copilot's `module:name`, as `dev:agent` runs `uvicorn agent.entrypoint:app`. If your build starts the Copilot some other way, ask Claude Code which `module:name` uvicorn needs to serve the Copilot's AgentCore app.

To check it, run this in a terminal in the `undercover` folder, with your `module:name` in place of `<copilot-app>`. It installs only the packages Render will, so a missing one shows up here:

```
uv run --isolated --only-group runtime --only-group context uvicorn <copilot-app> --port 8090
```

It should start without errors, and `http://127.0.0.1:8090/ping` in your browser answers with `"status": "Healthy"`. Stop it with Ctrl+C.

### Create the service

In the [Render Dashboard](https://dashboard.render.com), choose **New**, then **Web Service**, and pick the same repository and branch as your app. Then fill in the form, with your `module:name` in place of `<copilot-app>`:

| Field | Value |
|---|---|
| Name | `undercover-copilot` |
| Region | The same as your app |
| Language | Python 3 |
| Build Command | `uv sync --frozen --only-group runtime --only-group context` |
| Start Command | `uv run --no-sync python -m context & uv run --no-sync uvicorn <copilot-app> --host 0.0.0.0 --port $PORT` |
| Instance Type | Free |

Under **Advanced**, set the **Health Check Path** to `/ping`.

These are the agent's commands from Day 4, with the Copilot's app in place of the agent's. The Copilot's packages are in the `runtime` group with the claim agent's, since on the AWS route both runtimes deploy from one package. The context layer runs in the background inside the service, at `http://127.0.0.1:8006/mcp`, where the Copilot looks for it by default, as the agent does in its own service.

Under **Environment Variables**, add these lines:

| Key | Value |
|---|---|
| `OPENROUTER_API_KEY` | From your `.env` |
| `DATABASE_URL` | The same session pooler address as in your `.env` |
| `UNDERCOVER_API` | Your app's address on Render, such as `https://undercover.onrender.com` |

The Copilot doesn't get the agent's key, and unlike the agent it doesn't ask for one, so anyone can reach its address. The handler's sign-in is what protects it. Everything the Copilot does carries the sign-in of the handler who asked, and the app checks it on every call. Each turn starts by asking the app who is signed in, and a message without a valid sign-in ends there, before any call to the model.

With no memory setting, the Copilot's memory is off, as the claim agent's is on the free route.

Choose **Create Web Service**, and when the deploy is live, `https://<your-copilot>.onrender.com/ping` answers with `"status": "Healthy"`.

Render's documentation: [Web services](https://render.com/docs/web-services), [Environment variables and secrets](https://render.com/docs/configure-environment-variables), [Health checks](https://render.com/docs/health-checks).

## Step 3: Point the app at the Copilot

Open your app's web service, and under **Environment**, add:

| Key | Value |
|---|---|
| `UNDERCOVER_COPILOT` | Your Copilot's address on Render, such as `https://undercover-copilot.onrender.com` |

Choose **Save and deploy**. The app now sends each message to the Copilot on Render, with the handler's sign-in, as it does to the Copilot on your laptop.

A free service spins down after 15 minutes without a request, and the first request after that takes about a minute. So before you start a conversation, open the `/ping` addresses of the Copilot and the agent in your browser and wait until both answer. If you leave the conversation for more than 15 minutes, open both again before you carry on.

Render's documentation: [Free instances](https://render.com/docs/free).

## Step 4: The handover conversation

On the AWS route, the conversation starts with a dispatch of the whole queue, which needs more requests than a day's free allowance. On the free route we work two of the approvals over the limit with the agent on your laptop, then ask the Copilot about them. In a terminal in the `undercover` folder, reseed and start everything:

```
npm run seed
npm run dev
```

In a second terminal, work two claims:

```
uv run -m agent CLM-1003
uv run -m agent CLM-1011
```

If the agent approves a claim, it waits for sign-off. If it declines or refers one of them instead, work CLM-1008 as well. Then stop `npm run dev` with Ctrl+C. The rest happens on Render, and Supabase's free plan allows only 15 database connections at once.

Open your app on Render and sign in as Dana Whitfield, who can sign off over the limit. The Copilot's sidebar is open on the right. In one conversation, ask it these four questions, one at a time:

1. **"What needs me?"** The Copilot looks up the queue and names the payouts waiting for a sign-off.
2. **"Why does CLM-1011 need me?"**, or another claim it named. Watch the cards: the claim, the agent's steps on it and any lookups in the context layer.
3. **"Send it back with a note"**, saying what to look at again. On CLM-1011, for example, Meridian Air has already paid the customer $350 for the lost bag. The app sends it back as Dana and asks the agent on Render to resume the claim, which carries on from where your laptop left it in Supabase.
4. **"Sign off on another one."** The Copilot picks another payout waiting for a sign-off, and its graph pauses before it pays, with the confirmation card in the sidebar. Press Sign off and pay. The app sends Dana's answer to the Copilot, which resumes the conversation from its checkpoint and signs off as Dana.

Then restart the Copilot: on its **Deploys** page in the Render Dashboard, choose **Manual Deploy** and **Restart service**. When it's back, reload the dashboard, open the conversation from the sidebar's history and ask a follow-up, such as "Which claims have we dealt with?". The conversation's state is in Supabase, so the Copilot carries on where it left off.

The main lab's fifth question, "Are we meeting the success criteria we agreed?", runs an evaluation of the whole held-out set, which also needs more than the free allowance. If you have credit on OpenRouter, you can have the main lab's whole conversation instead, starting with a dispatch on the agent's service.

## Step 5: Compare with the AWS route

On the AWS route, each conversation has a Runtime session in a microVM of its own. If the handler answers the confirmation card after that session has ended, the Copilot picks up its pause in a new one. On Render the Copilot is one process that stays up between requests and sleeps after 15 minutes without one. Both keep the conversation in Supabase, which is why the restart lost nothing.

The AWS route also gives the Copilot an AgentCore Memory with a namespace for each handler, so it remembers what each handler has told it from one conversation to the next. Watch Part 6 of the main lab for the memory.

### When you've finished

Your three services on Render cost nothing, so you can keep them as long as you like. When you no longer need them, delete each one from its **Settings** page in the Render Dashboard.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Build your own LangGraph solution for a business problem where agents make decisions within limits, like approving a claim. Give it decisions with limits, where a person signs off above the limit; durable state, with LangGraph checkpoints in a database; an enterprise context layer for the agent; a memory with a namespace for each user, which on the free route can be LangGraph's <code>PostgresStore</code> in your Supabase database, with pgvector to search it; and deploy it, on Render on the free route. It can be a variation of UnderCover or something completely different.<br/><br/>I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>